# Foundry `gpt-5.4` · 응답 `usage` 로 토큰 실측하기

모델이 돌려주는 `usage` 를 직접 읽어서 **무엇이 얼마나 과금되는지** 확인합니다. 추정하지 않습니다.

### 확인할 것

| # | 주제 | 결론 미리보기 |
|---|---|---|
| 1 | `usage` 스키마 | API 마다 **필드 이름이 다릅니다** |
| 2 | `previous_response_id` | **전송량**만 줄여줍니다. 토큰 비용은 그대로입니다 |
| 3 | 프롬프트 캐시 | **최소 1,024토큰 + 접두부 완전 일치**가 필요합니다 |
| 4 | 가변값 배치 | 앞에 두면 **0%**, 뒤에 두면 **95%** |
| 5 | 압축의 함정 | 토큰이 줄어도 **비용은 오를 수 있습니다** |

### 준비물

- **인증** — `az login` (API 키 대신 Entra ID 토큰을 씁니다. 키를 파일에 남기지 않습니다)
- **커널** — `Python 3.12 (.venv · token-compression)`
- **설정** — 이 폴더의 `.env` · 없으면 `cp .env.example .env`

> **`No such file or directory: 'az'`** 가 나오면 az 가 없는 게 아니라 **커널이 경로를 못 찾는 것**입니다.
> macOS GUI 앱(VS Code)은 로그인 셸의 `PATH` 를 물려받지 않기 때문입니다.
> 2절에서 흔한 설치 위치를 직접 탐색하지만, 그래도 안 되면 `.env` 에 `AZ_CLI=/전체/경로/az` 를 넣어주세요.

## 1. 설정

엔드포인트와 배포명을 **코드에 넣지 않고 `.env` 에서 읽습니다.**

- 리소스명이 노트북 출력이나 커밋에 남지 않습니다
- 다른 구독이나 배포로 바꿀 때 코드를 건드리지 않아도 됩니다
- `override=False` 로 두어 **셸 환경변수가 `.env` 보다 우선**하게 했습니다 (CI 에서 덮어쓰기 편합니다)

In [ ]:
import json, os, subprocess, sys, time
import urllib.request, urllib.error
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

# 노트북 공용 유틸 (같은 폴더의 nbtools.py)
#   Usage      : 공급자별 usage 스키마를 하나로 정규화
#   Price      : 캐시 할인까지 반영한 비용 계산
#   show_table : 한글 정렬이 깨지지 않는 표 출력
from nbtools import Usage, Price, show_table

ENV_PATH = find_dotenv(usecwd=True) or str(Path.cwd() / ".env")
loaded = load_dotenv(ENV_PATH, override=False)


def require(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(f"{name} 가 없습니다. `cp .env.example .env` 후 값을 채우세요.")
    return v


ENDPOINT    = require("AZURE_OPENAI_ENDPOINT").rstrip("/")
DEPLOYMENT  = require("AZURE_OPENAI_DEPLOYMENT")
API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21")


def mask_endpoint(url):
    """출력에 리소스명이 그대로 남지 않게 가린다."""
    scheme, rest = url.split("://", 1)
    host, _, _ = rest.partition("/")
    name, _, domain = host.partition(".")
    return f"{scheme}://{name[:5]}***.{domain}"


print(".env       :", ENV_PATH if loaded else "(로드 실패 — 셸 환경변수 사용)")
print("엔드포인트 :", mask_endpoint(ENDPOINT))
print("배포명     :", DEPLOYMENT)
print("API 버전   :", API_VERSION)

## 2. 인증

- 기본은 **Entra ID** 입니다 — `az account get-access-token`
- `AZURE_OPENAI_API_KEY` 가 설정돼 있으면 그쪽을 우선 사용합니다
- `az` 실행 파일은 `PATH` 에 의존하지 않고 **흔한 설치 위치를 직접 탐색**합니다

In [ ]:
import shutil

SCOPE = "https://cognitiveservices.azure.com/.default"

# macOS GUI 앱(VS Code)은 로그인 셸의 PATH 를 물려받지 않습니다.
# 그래서 커널 안에서는 subprocess 가 'az' 를 못 찾는 일이 흔합니다.
AZ_CANDIDATES = [
    os.environ.get("AZ_CLI"),                       # .env 로 직접 지정 가능
    shutil.which("az"),
    "/opt/homebrew/bin/az",                         # Homebrew (Apple Silicon)
    "/usr/local/bin/az",                            # Homebrew (Intel)
    str(Path.home() / ".local/bin/az"),             # pipx / pip --user
    "/opt/az/bin/az",
]


def find_az():
    for c in AZ_CANDIDATES:
        if c and Path(c).exists():
            return c
    raise RuntimeError(
        "az CLI 를 찾지 못했습니다.\n"
        "  1) 설치 확인 : 터미널에서 `which az`\n"
        "  2) 경로 지정 : .env 에 AZ_CLI=/전체/경로/az 추가\n"
        "  3) 또는 API 키 사용 : .env 의 AZURE_OPENAI_API_KEY 채우기"
    )


def auth_headers():
    key = os.environ.get("AZURE_OPENAI_API_KEY")
    if key:
        return {"api-key": key}
    az = find_az()
    r = subprocess.run([az, "account", "get-access-token", "--scope", SCOPE, "-o", "json"],
                       capture_output=True, text=True, timeout=90)
    if r.returncode != 0:
        raise RuntimeError(f"az 토큰 발급 실패 — `az login` 이 필요할 수 있습니다.\n{r.stderr.strip()[:300]}")
    return {"Authorization": "Bearer " + json.loads(r.stdout)["accessToken"]}


HEADERS = auth_headers()
print("인증 방식:", "API 키" if "api-key" in HEADERS else f"Entra ID ({find_az()})")

## 3. 호출 함수

SDK 없이 `urllib` 로 직접 호출합니다.

- **이유** — SDK 를 쓰면 `usage` 가 객체로 감싸져 **실제 필드 구조가 보이지 않습니다**
- 두 API 를 모두 준비합니다 (`chat` / `responses`). 뒤에서 `usage` 모양을 비교할 예정입니다

In [ ]:
def chat(messages, deployment=DEPLOYMENT, **params):
    """Chat Completions 호출. (응답 dict, 소요초)"""
    url = f"{ENDPOINT}/openai/deployments/{deployment}/chat/completions?api-version={API_VERSION}"
    return _post(url, {"messages": messages, **params})


def responses(input_text, deployment=DEPLOYMENT, **params):
    """Responses API 호출. 경로와 페이로드 모양이 다르다."""
    url = f"{ENDPOINT}/openai/v1/responses?api-version=preview"
    return _post(url, {"model": deployment, "input": input_text, **params})


def _post(url, payload):
    req = urllib.request.Request(
        url, data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json", **HEADERS}, method="POST")
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            return json.loads(r.read().decode()), time.time() - t0
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8','replace')[:500]}")


def text_of(resp):
    """Chat Completions 응답 본문"""
    return (resp["choices"][0]["message"].get("content") or "").strip()


def rtext(resp):
    """Responses API 응답 본문"""
    out = []
    for item in resp.get("output", []):
        for c in item.get("content", []):
            if c.get("type") in ("output_text", "text"):
                out.append(c.get("text", ""))
    return "".join(out).strip()

## 4. 첫 호출 — 원시 `usage` 확인

가공 없이 **날것 그대로** 봅니다. 여기가 모든 판단의 출발점입니다.

In [ ]:
PROMPT = (
    "다음 약관에서 환불 수수료율과 면제 조건만 한 문장으로 답하라.\n\n"
    "제7조 환불 시 결제금액의 10%를 수수료로 공제한다. "
    "제8조 회사는 사전 고지 후 약관을 변경할 수 있다. "
    "제9조 단, 결제 후 7일 이내 취소는 수수료를 면제한다."
)

resp, elapsed = chat([{"role": "user", "content": PROMPT}])

print(f"소요 {elapsed:.2f}s")
print("응답:", text_of(resp))
print("\n--- 원시 usage (Chat Completions) ---")
print(json.dumps(resp["usage"], ensure_ascii=False, indent=2))

### 필드별 의미

| 필드 | 의미 |
|---|---|
| `prompt_tokens` | 입력 토큰 **전체** (캐시 적중분 포함) |
| `prompt_tokens_details.cached_tokens` | 그중 캐시로 재사용된 분량 → **크게 할인됩니다** |
| `completion_tokens` | 생성 토큰 |
| `completion_tokens_details.reasoning_tokens` | 추론 모델의 **내부 사고** 토큰. 응답 텍스트에는 없지만 **출력으로 과금됩니다** |
| `latency_checkpoint` | Azure 전용 지연 계측값 (표준 OpenAI 스키마에는 없습니다) |

> **정가가 붙는 입력은 `prompt_tokens` 가 아닙니다.**
> `billed_input = prompt_tokens − cached_tokens` 가 실제 판단 기준입니다.

## 5. 정규화 — `Usage.from_response()`

공급자와 API 마다 필드 이름이 달라서 **하나의 모양으로 맞춰야** 비교가 가능합니다.

같은 폴더의 `nbtools.py` 파서가 다음을 모두 받습니다.

- Chat Completions
- Responses API
- Anthropic Messages
- Google Gemini

In [ ]:
u = Usage.from_response(resp, model=DEPLOYMENT)

print(json.dumps(u.to_dict(), ensure_ascii=False, indent=2))
print()
print(f"과금 입력 토큰 (billed_input) : {u.billed_input:,}")
print(f"캐시 적중률                   : {u.cache_hit_rate:.1%}")
print("\ncached_tokens 가 0 인 이유는 11절에서 다룬다 (프롬프트가 짧아 캐시 조건 미달).")

## 6. 같은 모델, 다른 API — Responses API

같은 배포(`gpt-5.4`)인데도 **`usage` 필드 이름이 다릅니다.**

| 항목 | Chat Completions | Responses API |
|---|---|---|
| 입력 | `prompt_tokens` | `input_tokens` |
| 출력 | `completion_tokens` | `output_tokens` |
| 캐시 적중 | `prompt_tokens_details.cached_tokens` | `input_tokens_details.cached_tokens` |
| 캐시 쓰기 | — | **`input_tokens_details.cache_write_tokens`** |
| 추론 토큰 | `completion_tokens_details.reasoning_tokens` | `output_tokens_details.reasoning_tokens` |

**Responses API 에만 있는 기능**

- `previous_response_id` — 대화 이력을 서버가 보관합니다
- `prompt_cache_key` — 프롬프트 캐시 적중을 유도합니다

In [ ]:
rresp, rel = responses(PROMPT)

print(f"소요 {rel:.2f}s")
print("응답:", rtext(rresp)[:150])
print("\n--- 원시 usage (Responses API) ---")
print(json.dumps(rresp["usage"], ensure_ascii=False, indent=2))

### 파서가 두 스키마를 같게 만드는지 확인

**비교할 수 있는 것과 없는 것을 구분해야 합니다.**

| 필드 | 결정적인가 | 비교 대상 |
|---|---|---|
| `input_tokens` | 같은 입력이면 항상 같은 값 | **예** |
| `output_tokens` | 생성은 확률적이라 매번 다름 | 아니오 |

두 API 를 각각 호출했으므로 답변 문장이 조금씩 다르고, 그러면 출력 토큰 수도 달라집니다.
**정규화가 잘못된 것이 아닙니다.**

In [ ]:
cu = Usage.from_response(resp,  model=DEPLOYMENT)   # 4절 Chat Completions
ru = Usage.from_response(rresp, model=DEPLOYMENT)   # 6절 Responses

# ── (1) 원본 필드 이름은 다르다 ────────────────────────────────────
print("원본 usage 의 최상위 필드 이름")
print(f"  chat      : {sorted(resp['usage'].keys())}")
print(f"  responses : {sorted(rresp['usage'].keys())}")

# ── (2) 정규화 후 키는 같아야 한다 — 이게 파서의 역할 ──────────────
print(f"\n정규화 후 키가 동일한가: {cu.to_dict().keys() == ru.to_dict().keys()}")

# ── (3) 입력 계열: 결정적이므로 값까지 같아야 한다 ─────────────────
print("\n[입력 계열] 같은 프롬프트 → 값이 같아야 정상")
print(f"  {'':22s} {'chat':>18s} {'responses':>12s}   일치")
for k in ("input_tokens", "cached_tokens", "cache_write_tokens", "billed_input"):
    a, b = cu.to_dict()[k], ru.to_dict()[k]
    print(f"  {k:22s} {str(a):>18s} {str(b):>12s}   {'O' if a == b else 'X'}")

# ── (4) 출력 계열: 비결정적이므로 달라도 정상 ──────────────────────
print("\n[출력 계열] 생성 결과라 매번 다름 → 값이 달라도 정상")
print(f"  {'':22s} {'chat':>18s} {'responses':>12s}")
for k in ("output_tokens", "reasoning_tokens"):
    print(f"  {k:22s} {str(cu.to_dict()[k]):>18s} {str(ru.to_dict()[k]):>12s}")

ans_c, ans_r = text_of(resp), rtext(rresp)
print(f"\n  실제 답변을 보면 이유가 보인다 ({len(ans_c)}자 vs {len(ans_r)}자)")
print(f"    chat      : {ans_c}")
print(f"    responses : {ans_r}")

print(f"""
{'schema':22s} {cu.schema:>18s} {ru.schema:>12s}
=> 필드 이름이 달라도 정규화 후에는 같은 키로 접근할 수 있다. 이것이 파서의 목적이다.
   비교 실험을 할 때는 API 를 섞지 말고 하나로 고정하세요.""")

## 7. `previous_response_id` — 보내는 양과 과금되는 양은 다릅니다

2턴에서 **짧은 질문만 보내고** 이전 대화는 id 로 참조합니다.
전송하는 텍스트는 줄지만 **과금되는 `input_tokens` 는 줄지 않습니다.**
서버가 이력을 다시 붙여서 모델에 넣기 때문입니다.

**같은 2턴을 두 방식으로 보내 비교합니다.**

| | 2턴에 보내는 것 |
|---|---|
| **A** | 후속 질문만 + `previous_response_id` |
| **B** | 1턴 질문 + 1턴 답변 + 후속 질문 (직접 이어붙임) |

A 가 훨씬 적게 보내는데도 과금 `input_tokens` 가 B 와 비슷하다면,
`previous_response_id` 는 **전송량만** 줄이는 장치라는 뜻입니다.

In [ ]:
# ── 1턴 ────────────────────────────────────────────────────────────
t1, _ = responses(PROMPT)
u1 = Usage.from_response(t1, model=DEPLOYMENT)
a1 = rtext(t1)

show_table(
    ["1턴", "값"],
    [["보낸 텍스트",   f"{len(PROMPT):,}자 (약관 프롬프트)"],
     ["input_tokens",  f"{u1.input_tokens:,}"],
     ["output_tokens", f"{u1.output_tokens:,}"],
     ["응답",          a1[:52] + "…"]],
    align=["left", "left"],
    title="【1턴】",
)

# ── 2턴을 두 방식으로 ───────────────────────────────────────────────
FOLLOWUP = "그럼 7일이 지나면 얼마를 돌려받나?"
# 참고: 이 응답은 서버에 기본 30일 보관된다(store 기본값 true).
#       저장을 원치 않으면 store=False (단, 체이닝 불가)
#       즉시 지우려면 DELETE /openai/v1/responses/{id}
MANUAL = f"{PROMPT}\n\n[답변] {a1}\n\n{FOLLOWUP}"

t2a, _ = responses(FOLLOWUP, previous_response_id=t1["id"])   # A: 서버가 이력 보관
u2a = Usage.from_response(t2a, model=DEPLOYMENT)

t2b, _ = responses(MANUAL)                                     # B: 이력을 직접 전송
u2b = Usage.from_response(t2b, model=DEPLOYMENT)

show_table(
    ["2턴 방식", "보낸 내용", "보낸 문자수", "과금 input", "output"],
    [["A) previous_response_id", "후속질문만",
      f"{len(FOLLOWUP):,}", f"{u2a.input_tokens:,}", f"{u2a.output_tokens:,}"],
     ["B) 이력을 직접 전송", "1턴질문+답변+후속질문",
      f"{len(MANUAL):,}", f"{u2b.input_tokens:,}", f"{u2b.output_tokens:,}"]],
    foot=["차이 (B − A)", "",
          f"{len(MANUAL)-len(FOLLOWUP):+,}", f"{u2b.input_tokens-u2a.input_tokens:+,}", ""],
    align=["left", "left", "right", "right", "right"],
    title="【2턴】 같은 질문을 두 방식으로 보낸다",
    note=f'A 가 실제로 보낸 것: "{FOLLOWUP}" — 이전 대화는 한 글자도 안 보냈다.',
)

print(f"""A 는 B 보다 {len(MANUAL)/len(FOLLOWUP):.0f}배 적게 보냈는데, 과금되는 input 은 오히려 {u2a.input_tokens-u2b.input_tokens:+,} 토큰입니다.
=> previous_response_id 는 '전송량'을 줄이는 장치이지 '토큰 비용'을 줄이는 장치가 아닙니다.""")

### 그 숫자는 어떻게 나왔을까요 — 분해

2턴 `input_tokens` 가 커진 이유를 **실측값만으로** 설명합니다.

- 후속 질문을 **단독으로 한 번 더 호출**해서 그 자체의 토큰 수를 잽니다
- 1턴 입력 + 1턴 출력 + 후속질문 = 2턴 입력 인지 확인합니다
- 차이가 **0** 이면 완전히 설명된 것입니다

In [ ]:
# 후속 질문만 단독으로 보내서 그 자체의 토큰 수를 실측
tf, _ = responses(FOLLOWUP)
uf = Usage.from_response(tf, model=DEPLOYMENT)

parts = [
    ("1턴 입력 (약관 프롬프트)", u1.input_tokens,  "1턴 응답의 input_tokens"),
    ("1턴 출력 (모델 답변)",     u1.output_tokens, "1턴 응답의 output_tokens"),
    ("2턴 후속질문",             uf.input_tokens,  "후속질문만 단독 호출한 input_tokens"),
]
subtotal = sum(v for _, v, _ in parts)

show_table(
    ["구성 요소", "토큰", "측정 방법"],
    [[n, f"{v:,}", how] for n, v, how in parts],
    foot=["합계", f"{subtotal:,}", ""],
    align=["left", "right", "left"],
    title="2턴에서 모델이 실제로 읽은 것 = 지금까지의 대화 전부",
)

show_table(
    ["검증", "토큰", ""],
    [["실측 2턴 input_tokens", f"{u2a.input_tokens:,}", ""],
     ["분해 합계",             f"{subtotal:,}",         ""],
     ["차이",                  f"{u2a.input_tokens - subtotal:+,}", "0 이면 완전히 설명된 것"]],
    align=["left", "right", "left"],
)

print(f"""보낸 텍스트는 {len(FOLLOWUP)}자짜리 질문 하나뿐이지만,
서버가 1턴 질문({u1.input_tokens}) + 1턴 답변({u1.output_tokens}) 을 다시 앞에 붙여
총 {u2a.input_tokens}토큰을 모델에 넣고 그만큼 과금한다.""")

### 보관 기간과 삭제

서버가 대화를 들고 있다는 건 **어딘가에 저장된다**는 뜻입니다.

| 항목 | 값 |
|---|---|
| 기본 보관 | **30일** (`store` 기본값 `true`) |
| 기간 연장 | **불가** — 늘리는 파라미터가 없습니다 |
| 저장 안 하기 | `"store": false` |
| 즉시 삭제 | `DELETE /openai/v1/responses/{response_id}` |

```python
responses(PROMPT, store=False)      # 저장 안 함 (체이닝 포기)
# DELETE {ENDPOINT}/openai/v1/responses/{id}?api-version=preview
#   -> {"object": "response.deleted", "deleted": true}
```

**직접 확인한 동작 두 가지**

- **`store=false` 는 체이닝이 안 됩니다** — 그 id 로 `previous_response_id` 를 걸면 `400` 이 납니다.
  서버측 상태와 무저장은 **양자택일**입니다.
- **삭제해도 체이닝은 한동안 동작합니다** — `DELETE` 직후 `GET` 은 `404` 인데
  같은 id 로 체이닝하면 **여전히 `200`** 이었습니다(30초 뒤까지 동일했습니다).
  삭제는 *조회 가능성*을 없애는 것이지 대화 상태가 즉시 사라지는 것이 아닙니다.

> 30일 보관이 부담스러운 데이터라면 **`store=false` + 이력 직접 전송(B 방식)** 이 안전합니다.

### 턴이 쌓이면 어떻게 될까요

- 보내는 질문 길이는 **일정하게** 유지합니다
- 그런데 매 턴 `input_tokens` 는 **계속 자랍니다**
- 이것이 대화가 '곱셈으로 부푸는' 이유입니다

In [ ]:
FOLLOWUPS = [
    "그럼 7일이 지나면 얼마를 돌려받나?",
    "10만원을 결제했다면 실제 환불액은?",
    "그 금액에서 부가세는 어떻게 되나?",
]

r0, _ = responses(PROMPT)
h0 = Usage.from_response(r0, model=DEPLOYMENT)
prev, total = r0["id"], h0.input_tokens
rows = [["1", f"{len(PROMPT):,}", f"{h0.input_tokens:,}", f"{h0.output_tokens:,}", f"{total:,}"]]
hist = [h0.input_tokens]

for i, q in enumerate(FOLLOWUPS, start=2):
    r, _ = responses(q, previous_response_id=prev)
    h = Usage.from_response(r, model=DEPLOYMENT)
    prev, total = r["id"], total + h.input_tokens
    hist.append(h.input_tokens)
    rows.append([str(i), f"{len(q):,}", f"{h.input_tokens:,}", f"{h.output_tokens:,}", f"{total:,}"])
    time.sleep(1)

show_table(
    ["턴", "보낸 문자수", "input", "output", "누적 input"],
    rows,
    align=["right"] * 5,
    note=f"보낸 문자수는 턴마다 20자 안팎으로 일정한데 input 토큰은 {hist[0]} → {hist[-1]} 로 늘었다.",
)

print(f"""{len(hist)}턴 대화의 누적 과금 입력은 {total:,} 토큰입니다.
이것이 대화가 '곱셈으로 부푸는' 이유이고, 컨텍스트 압축이 필요한 지점이다.""")

## 8. 프롬프트 캐시 — `cached_tokens` 가 0이 아니려면

지금까지 `cached_tokens` 가 계속 **0** 이었습니다. 캐시가 없어서가 아니라 **조건 미달**이었습니다.

**적중 조건 두 가지**

1. 프롬프트가 **최소 1,024토큰** 이상이어야 합니다
2. **앞의 1,024토큰이 완전히 동일**해야 합니다

앞서 쓴 프롬프트는 100토큰 남짓이라 1번부터 넘지 못했습니다.

> **재현성 주의** — 캐시는 셀을 다시 실행해도 남아 있어서, 두 번째 실행부터는
> 1회차가 이미 적중해 버립니다. 그래서 실행할 때마다 **`RUN_ID`** 를 접두부 맨 앞에 붙여
> 매번 '한 번도 본 적 없는 프롬프트'로 시작합니다.
> Azure 프롬프트 캐시는 사용자가 직접 삭제할 수 없기 때문입니다.

In [ ]:
import uuid

# ── 재실행해도 항상 '찬 캐시'에서 시작하게 만든다 ──────────────────────
# Azure 프롬프트 캐시는 사용자가 지울 수 없습니다. 대신 실행할 때마다
# 접두부 맨 앞에 새 RUN_ID 를 붙여 '한 번도 본 적 없는 프롬프트'로 만듭니다.
# RUN_ID 는 이 셀 안에서는 고정이라 2·3회차 적중은 정상적으로 관찰됩니다.
RUN_ID = uuid.uuid4().hex[:8]
print(f"RUN_ID = {RUN_ID}  (셀을 다시 실행하면 새 값 -> 1회차는 항상 0%)\n")

BODY = "".join(
    f"제{i}조 회원은 서비스 이용 시 관련 법령과 본 약관을 준수하여야 하며, "
    f"위반 시 회사는 제{i}호에 따라 이용을 제한할 수 있다. 환불 수수료는 10%이다.\n"
    for i in range(1, 61)
)

# 1,024토큰을 확실히 넘기는 고정 접두부
CACHE_PREFIX = (
    f"[run={RUN_ID}] 당신은 통신사 고객센터 상담원입니다. 아래 약관을 근거로만 답하십시오.\n\n"
    + BODY
)

rows = []
for turn in (1, 2, 3):
    r, _ = chat(
        [{"role": "system", "content": CACHE_PREFIX},
         {"role": "user",   "content": "환불 수수료율은?"}],
        max_completion_tokens=16,
    )
    x = Usage.from_response(r, model=DEPLOYMENT)
    rows.append([str(turn), f"{x.input_tokens:,}", f"{x.cached_tokens:,}",
                 f"{x.billed_input:,}", f"{x.cache_hit_rate:.1%}"])
    time.sleep(2)

show_table(
    ["회차", "입력", "캐시적중", "과금입력", "적중률"],
    rows,
    align=["right"] * 5,
    title=f"동일한 프롬프트 {len(CACHE_PREFIX):,}자를 3번 반복 호출",
    note="적중 토큰이 128 의 배수로 떨어지는지 보라 — 캐시는 128토큰 단위로 끊깁니다.",
)

### 읽는 법

- **1회차는 항상 0** 입니다 — 캐시를 *채우는* 요청이기 때문입니다. 첫 요청은 늘 정가입니다
- **적중은 2회차, 때로는 3회차부터** 입니다 — 캐시 쓰기 반영에 시간차가 있습니다
  → 캐시 효과는 **1회 호출로 판단하면 안 됩니다**
- **적중분은 크게 할인됩니다** — Provisioned 는 최대 100% 입니다
- **적중 토큰은 128의 배수**입니다 — 캐시는 128토큰 단위로 끊기고, 자투리는 정가입니다

여기까지는 "잘 되는 경우"입니다. 압축에서 진짜 중요한 것은 **깨지는 경우**입니다.

### 가변값의 위치가 캐시를 살리기도 죽이기도 합니다

내용도 토큰 수도 같은데 **세션 ID 위치만** 바꿔 봅니다.

- 캐시는 **접두부 완전 일치**로 판정합니다
- 맨 앞 한 글자만 달라도, 뒤의 3,000토큰이 같아도 **전량 미스**입니다

> 캐시는 타이밍에 따라 흔들리므로 각 케이스를 **4회씩** 돌려 최고 적중률로 비교합니다.

In [ ]:
def cache_trial(tag, label, make_system, turns=4):
    """캐시는 타이밍 의존적이라 한두 번으로 판단하면 안 된다. 여러 번 돌려 최고치를 본다."""
    rows, hits = [], []
    for t in range(1, turns + 1):
        r, _ = chat(
            [{"role": "system", "content": make_system()},
             {"role": "user",   "content": "환불 수수료율은?"}],
            max_completion_tokens=16,
        )
        x = Usage.from_response(r, model=DEPLOYMENT)
        hits.append(x.cache_hit_rate)
        rows.append([f"{t}회차", f"{x.input_tokens:,}", f"{x.cached_tokens:,}",
                     f"{x.cache_hit_rate:.1%}"])
        time.sleep(1)
    show_table(["회차", "입력", "캐시적중", "적중률"], rows,
               foot=["최고", "", "", f"{max(hits):.1%}"],
               align=["left", "right", "right", "right"], title=label)
    return max(hits)


# 두 케이스 모두 RUN_ID 를 붙여 '찬 캐시'에서 출발합니다.
# 앞: 접두부가 매번 달라지므로 구조적으로 적중이 불가능
front = cache_trial("front", "가변값이 앞 — 세션ID를 맨 위에",
    lambda: f"[run={RUN_ID}/front][session={time.time_ns()}]\n당신은 상담원입니다.\n{BODY}")

# 뒤: 접두부가 고정이므로 적중할 수 있습니다
back = cache_trial("back", "가변값이 뒤 — 세션ID를 맨 아래",
    lambda: f"[run={RUN_ID}/back] 당신은 상담원입니다.\n{BODY}[session={time.time_ns()}]")

show_table(
    ["세션ID 위치", "최고 적중률"],
    [["맨 앞", f"{front:.1%}"], ["맨 뒤", f"{back:.1%}"]],
    title="결론",
    note="같은 내용, 같은 토큰 수. 위치만 다른데 결과가 갈립니다. "
         "'앞' 케이스는 몇 번을 돌려도 0% — 우연이 아니라 구조적으로 불가능합니다.",
)

### 압축 작업에 주는 함의

압축은 본질적으로 **텍스트를 바꾸는 일**이라 구조적으로 캐시와 충돌합니다.

| 원칙 | 이유 |
|---|---|
| 고정부(시스템·정책·few-shot)는 **건드리지 않습니다** | 접두부가 바뀌면 전량 미스입니다 |
| 압축 대상은 **가변부(뒤쪽)** 로 모읍니다 | 뒤쪽 변경은 앞의 캐시를 깨지 않습니다 |
| 배치는 **고정부 → 준고정부 → 가변부** 순입니다 | 캐시 가능한 접두부를 최대한 길게 가져갑니다 |
| **질문 조건부 압축은 캐시와 상충합니다** | 질문마다 접두부가 바뀌기 때문입니다 |

> LongLLMLingua 같은 질문 조건부 기법은 압축률이 가장 높지만 **캐시를 구조적으로 파괴합니다.**
> `labs/01` 에서 이 트레이드오프를 실측할 예정입니다.

## 9. 캐시 유지기간과 설정

**모델 세대에 따라 방식이 완전히 다릅니다.**

| | GPT-5.4 이하 | GPT-5.6 이상 |
|---|---|---|
| 파라미터 | `prompt_cache_retention` | `prompt_cache_options.ttl` |
| 의미 | **최대** 보관 정책 | **최소** 보장 수명 |
| 값 | `in_memory`(기본) / `24h` | `30m` (**유일한 값**, 기본) |
| 최대 유지 | 1시간 / **24시간** | 명시 없음 (최소 30분 보장) |
| 명시적 breakpoint | 미지원 (400) | `prompt_cache_breakpoint` 지원 |

**`in_memory` (기본값)**
- 비활성 **5~10분** 안에 대체로 삭제됩니다
- 마지막 사용 후 **1시간 안에는 반드시** 삭제됩니다
- GPT-4o 이상 전 모델이 지원합니다

**`24h` (확장 보존)**
- 최대 **24시간** 유지됩니다. KV 텐서를 GPU 로컬 스토리지로 내려 용량을 확보하는 방식입니다
- 지원 모델: `gpt-5.5`, `gpt-5.4`, `gpt-5.3-codex`, `gpt-5.2`, `gpt-5.1*`, `gpt-5`, `gpt-4.1` 등
- **가격은 `in_memory` 와 동일합니다** → 재사용 주기가 길면 켜는 쪽이 유리합니다
- `gpt-5.5` 는 기본으로 켜져 있고, `gpt-5.4` 이하는 기본이 `in_memory` 입니다

> 캐시는 **구독 간에 공유되지 않습니다.**

In [ ]:
def probe(label, extra):
    """이 배포가 어떤 캐시 파라미터를 받는지 확인한다."""
    try:
        r, _ = chat(
            [{"role": "system", "content": CACHE_PREFIX},
             {"role": "user",   "content": "환불 수수료율은?"}],
            max_completion_tokens=16, **extra,
        )
        x = Usage.from_response(r, model=DEPLOYMENT)
        return [label, "지원", f"cached={x.cached_tokens:,}"]
    except RuntimeError as e:
        return [label, "미지원", str(e).split(":")[0]]


show_table(
    ["파라미터", "결과", "비고"],
    [probe("prompt_cache_retention=in_memory", {"prompt_cache_retention": "in_memory"}),
     probe("prompt_cache_retention=24h",       {"prompt_cache_retention": "24h"}),
     probe("prompt_cache_options(ttl=30m)",    {"prompt_cache_options": {"mode": "implicit", "ttl": "30m"}})],
    align=["left", "left", "left"],
    title=f"배포 {DEPLOYMENT} 가 받는 캐시 파라미터",
    note="prompt_cache_retention 값을 바꾸면 캐시 파티션이 갈려 적중이 0으로 떨어질 수 있다. "
         "실험 중에는 한 값으로 고정하세요.",
)

### 추가로 알아둘 것

**`prompt_cache_key` (GPT-5.6 이상)**
- 같은 접두부를 공유하는 요청에 **같은 키**를 주면 매칭률이 올라갑니다
- 다만 **(접두부, 키) 조합이 분당 약 15회를 넘으면 일부 미스**가 납니다
- 트래픽이 많으면 키를 여러 개로 쪼개되, 키와 접두부의 매핑은 고정해야 합니다

**캐시 쓰기 요금 (GPT-5.6 이상)**
- 5.4 이하는 쓰기가 무료지만, **5.6부터는 쓰기에도 요금**이 붙습니다
- "매번 조금씩 다른 프롬프트"가 **이중 손해**입니다 — 읽기 할인은 못 받고 쓰기 요금은 계속 냅니다
- 앞서 본 `cache_write_tokens` 가 이 값입니다

## 10. 왜 압축률만 보면 안 될까요

- 압축은 텍스트를 바꿉니다 → **접두부가 바뀝니다** → **캐시가 깨집니다**
- 그래서 토큰을 절반으로 줄여도 **비용은 오를 수 있습니다**

아래는 8절에서 **실제로 측정한 캐시 적중 상태**를 기준으로 계산합니다.

In [ ]:
# 8절에서 캐시가 잘 맞던 상태를 다시 한 번 실측
r_cached, _ = chat(
    [{"role": "system", "content": CACHE_PREFIX},
     {"role": "user",   "content": "환불 수수료율은?"}],
    max_completion_tokens=16,
)
warm = Usage.from_response(r_cached, model=DEPLOYMENT)

# 같은 내용을 절반으로 '압축'했다고 가정 — 단 접두부가 바뀌어 캐시는 전량 미스
compressed = Usage(
    input_tokens=warm.input_tokens // 2,
    cached_tokens=0,
    output_tokens=warm.output_tokens,
)

# gpt-5.4 예시 단가 (USD / 1K 토큰) — 실제 값은 요금표로 교체할 것
price = Price(input_per_1k=0.00125, cached_input_per_1k=0.000125, output_per_1k=0.01)

d = compressed.cost(price) - warm.cost(price)
show_table(
    ["상태", "입력", "캐시적중", "과금입력", "비용(USD)"],
    [["압축 전 (캐시 O)", f"{warm.input_tokens:,}", f"{warm.cached_tokens:,}",
      f"{warm.billed_input:,}", f"${warm.cost(price):.5f}"],
     ["압축 후 (캐시 X)", f"{compressed.input_tokens:,}", f"{compressed.cached_tokens:,}",
      f"{compressed.billed_input:,}", f"${compressed.cost(price):.5f}"]],
    foot=["차이", f"{compressed.input_tokens-warm.input_tokens:+,}", "",
          f"{compressed.billed_input-warm.billed_input:+,}", f"${d:+.5f}"],
    align=["left", "right", "right", "right", "right"],
    note=f"토큰은 {(1 - compressed.input_tokens/warm.input_tokens):.0%} 줄었는데 "
         f"비용은 {d/warm.cost(price):+.0%} 늘었다.",
)

print("""캐시 적중률이 높은 구간을 압축하면 손해입니다.
=> 판단 기준은 압축률이 아니라 billed_input 이다.""")

## 11. 정리

| 배운 것 | 왜 중요한가 |
|---|---|
| `usage` 필드명은 API마다 다릅니다 | 비교하려면 **정규화가 먼저**입니다 |
| `cached_tokens` 를 빼야 과금액입니다 | 압축 효과는 **`billed_input`** 으로 판단합니다 |
| `reasoning_tokens` 는 안 보여도 과금됩니다 | 추론 모델은 출력 비용이 체감보다 큽니다 |
| `previous_response_id` 는 전송량만 줄입니다 | 비용을 줄이는 건 **캐시**입니다 |
| 응답은 **30일 보관**되고 연장이 안 됩니다 | 민감 데이터는 `store=false` 를 검토합니다 |
| 캐시는 **1,024토큰 + 접두부 일치**가 조건입니다 | 짧은 프롬프트에는 아예 발동하지 않습니다 |
| 가변값을 앞에 두면 **적중률 0%** 입니다 | **배치 순서가 비용을 좌우합니다** |
| **압축률과 비용 절감은 다릅니다** | 캐시를 깨면 토큰이 줄어도 비용은 오릅니다 |

### 다음 단계

골든셋(`shared/golden/`)을 만들어 **"무엇이 깨지면 실패인가"를 먼저 정의**합니다.

> 압축기를 만든 뒤에 기준을 정하면, 자기가 만든 압축기에 유리한 기준을 고르게 됩니다.